# Equal-Weight S&P 500 Index Fund

## Introduction & Library Imports

The objective of this core engineering module is to build an automated, programmatic asset-allocation engine. The application ingests a dynamic, user-specified portfolio valuation and algorithmically calculates the optimized integer-based share distribution required to construct a mathematically precise equal-weight replication of the S&P 500 index.

## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [18]:
import numpy as np
import pandas as pd
import requests
import xlsxwriter
import math
import yfinance as yf
import time

## Index Constituent Ingestion

In a production environment, index adjustments are managed via a live enterprise pipeline connected directly to index providers (e.g., S&P Dow Jones Indices). For this standalone workstation implementation, the pipeline ingests a local, standardized CSV manifest (sp_500_stocks.csv) representing the baseline index constituents to seed the execution array."

In [19]:
stocks = pd.read_csv('sp_500_stocks.csv')
ticker_list = stocks["Ticker"].tolist()

## Architectural API Migration Note

The baseline tutorial architecture originally utilized the legacy IEX Cloud endpoint layout. Following the full deprecation and retirement of that service, this data engineering pipeline was fully refactored to query real-time market cap arrays and financial vectors natively through the yfinance API engine, optimizing network reliability and eliminating legacy token dependencies."

In [32]:
import pandas as pd
import yfinance as yf

print("Initializing High-Volume Ingestion Matrix...")

# 1. Execute a single, vectorized batch query for all tickers simultaneously
# This reduces network I/O overhead and avoids rate-limiting blocks
batch_data = yf.download(ticker_list, period="1d", group_by="ticker", progress=True)

# 2. Stage a vector to compile structured record nodes rapidly
parsed_records = []

print("\nProcessing and sanitizing financial vectors...")

for ticker in ticker_list:
    try:
        # Isolate the data block for the target security symbol
        if ticker in batch_data.columns.levels[0]:
            ticker_df = batch_data[ticker]
            
            # Extract closing price safely from the most recent historical matrix row
            if not ticker_df.empty:
                price = float(ticker_df['Close'].iloc[-1])
                
                # Retrieve structural market cap from the Ticker metadata engine
                market_cap = yf.Ticker(ticker).info.get('marketCap', None)
                
                # Append cleansed structural nodes to our array staging area
                parsed_records.append({
                    "Ticker": ticker,
                    "Stock Price": price,
                    "Market Capitalization": market_cap,
                    "Number of Shares to Buy": "N/A"
                })
    except Exception as e:
        # Gracefully trap structural missing elements without crashing execution
        continue

# 3. Instantiate the production DataFrame using structured records arrays
final_dataframe = pd.DataFrame(parsed_records)
print(f"Data ingestion complete! Target matrix established with {len(final_dataframe)} active constituents.")
final_dataframe

Initializing High-Volume Ingestion Matrix...


[****                   9%                       ]  43 of 505 completed$HFC: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
[****                   9%                       ]  44 of 505 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}
[*****                 10%                       ]  52 of 505 completed$GPS: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
[*****                 10%                       ]  53 of 505 completed$WBA: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
[******                12%                       ]  62 of 505 completed$K: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")
[*******               15%                       ]  74 of 505 compl


Processing and sanitizing financial vectors...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HBI"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IPG"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}


Data ingestion complete! Target matrix established with 420 active constituents.


,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy
0,A,135.529999,3.824903e+10,N/A
1,AAL,14.640000,9.682679e+09,N/A
2,AAP,60.240002,3.634232e+09,N/A
3,AAPL,312.059998,4.583336e+12,N/A
4,ABBV,217.720001,3.846661e+11,N/A
...,...,...,...,...
415,SNPS,475.619995,9.107140e+10,N/A
416,SO,92.050003,1.037681e+11,N/A
417,SPG,204.910004,7.786163e+10,N/A
418,SPGI,424.000000,1.255040e+11,N/A


## Algorithmic Allocation Matrix

Cleanses the real-time pricing dataframe by filtering missing elements, determines individual target capital allocations per stock based on the dynamic portfolio valuation input, and applies a strict mathematical floor calculation (math.floor) to ensure allocations are limited to tradable, whole integer-based share counts."

In [25]:

#get someones value of their portfolio
portfolio_size = input('Enter the value of your portfolio:')

#force input to be a float variable
try:
    val = float(portfolio_size)
except ValueError:
    print("That's not a number! \nPlease try again:")
    portfolio_size = input('Enter the value of your portfolio:')
    val = float(portfolio_size)

#if a string is input twice, an error will occur

Enter the value of your portfolio: 1000000


In [26]:
import math

#Clean up the dataframe by dropping any row that has a blank (NaN) Stock Price
final_dataframe = final_dataframe.dropna(subset=['Stock Price'])

#Run your position size calculation
position_size = val / len(final_dataframe.index)

#Loop through and calculate the number of shares securely using the remaining index labels
for i in final_dataframe.index:
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / final_dataframe.loc[i, 'Stock Price'])

#View your updated dataframe
final_dataframe

#math.floor is to round down in the case that brokers don't accpet fractional trading

,Ticker,Stock Price,Market Capitalization,Number of Shares to Buy
0,A,135.53,38249025536,16
1,AAL,14.64,9682678784,151
2,AAP,60.24,3634231552,36
3,AAPL,312.06,4583336181760,7
4,ABBV,217.72,384666140672,10
...,...,...,...,...
500,YUM,147.95,40778158080,14
501,ZBH,82.33,15927766016,26
502,ZBRA,243.63,11604923392,9
503,ZION,62.45,9185884160,35


## Automated Executive Sheet Compiling

Utilizes the XlsxWriter compiler engine to override default unformatted spreadsheet templates. This process maps the raw dataframe into a highly polished, production-ready spreadsheet.

Workbook Initialization: Establishes the underlying file writing stream targets."

In [27]:
writer = pd.ExcelWriter('recommended trades.xlsx', engine = 'xlsxwriter')
final_dataframe.to_excel(writer, 'Recommended Trades', index = False)

#using pandas library and not xlsxwriter as pandas is tightly integrated to excel
#the engine = 'xlsxwriter' may seem redundant but the library can be used to save XML files hence need to specify

C:\Users\nicks\AppData\Local\Temp\ipykernel_3496\949049150.py:2: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  final_dataframe.to_excel(writer, 'Recommended Trades', index = False)


## UI/UX Formatting Blobs

Establishes explicit structural formats, assigning custom hex values (#0a0a23 background / #ffffff foreground text matrices), structural cell borders, and currency masks.

DRY-Compliant Layout Automation: To respect the 'Don't Repeat Yourself' (DRY) architectural paradigm, column tracking and custom typography applications are compiled dynamically through key-value loops rather than manual cell formatting statements."

In [28]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_format = writer.book.add_format(
    {
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)

dollar_format = writer.book.add_format(
    {
        'num_format': '$0.00',
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)

integer_format = writer.book.add_format(
    {
        'num_format': '0',
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)


In [29]:
# writer.sheets['Recommended Trades'].set_column('A:A', 18, string_format)
# writer.sheets['Recommended Trades'].set_column('B:B', 18, string_format)
# writer.sheets['Recommended Trades'].set_column('C:C', 18, string_format)
# writer.sheets['Recommended Trades'].set_column('D:D', 18, string_format)
# writer.close()

writer.sheets['Recommended Trades'].write('A1', 'Ticker', string_format)
writer.sheets['Recommended Trades'].write('B1', 'Stock Price', dollar_format)
writer.sheets['Recommended Trades'].write('C1', 'Market Capitalization', dollar_format)
writer.sheets['Recommended Trades'].write('D1', 'Number of Shares to Buy', integer_format)

0

In [30]:
column_formats = {
    'A': ['Ticker', string_format],
    'B': ['Stock Price', dollar_format],
    'C': ['Market Capitalization', dollar_format],
    'D': ['Number of Share to Buy', integer_format]
}

#keys method returns all of the variables on top
for column in column_formats.keys():
    writer.sheets['Recommended Trades'].set_column(f'{column}:{column}', 18, column_formats[column][1])
    writer.sheets['Recommended Trades'].write(f'{column}1', column_formats[column][0], column_formats[column][1])



## Data Buffer Finalization

Explicitly executes writer.close() to close data storage buffers, seal the file architecture, and release operating system file-system locks cleanly."

In [31]:
writer.close()